# Gauss-Seidel MDA: converging the models without the optimiser

An MDA (multidisciplinary analysis) converges the coupled models at fixed inputs, with
no optimiser involved. At the end, every quantity that feeds back into an earlier model
equals what the models compute from it. PROCESS does this with its idempotence loop.
`Caller.call_models` repeats the full model sequence in call order until the objective
and the constraints stop changing, at most ten passes. That is a Gauss-Seidel iteration:
the models are swept in order and each uses the newest values. One PROCESS pass is one
sweep.

Here the same iteration is built on the graph of the models. The graph knows which
models feed back into which, so only those are swept and the rest run once. Each
coupled group gets three steps:

1. **Cut** (`FixedPointCut`). Some variables are read by an earlier model in the call
   order than the one that computes them. Each such reader gets a copy, `^hat.x`,
   instead. The group is now a straight sequence with one requirement attached: the
   copy must equal the computed value. That requirement is a fixed-point problem.
2. **Nest** (`Nest`). Some models contain a solve of their own, such as the root find in
   the coil sizing or the iteration for the ion temperature. These are left as they are
   and converged inside each sweep, as inside a PROCESS pass.
3. **Assign a driver**. A driver is the algorithm that solves a problem. Here it is a
   fixed-point iteration (`PicardDriver`): sweep until the copies stop changing, as
   PROCESS's loop does.

Which variables to copy is a choice. Jacobi copies every variable that crosses between
models; every model then reads only last sweep's values, so the group could run in
parallel. Gauss-Seidel copies only what is read before it is computed in a given order.
"Minimal" is Gauss-Seidel in the order that needs the fewest copies. All three are
implemented in `functional_process.cottax.recipes`. The hand-picked cut in `mda.CUTS`,
used by the MDF, IDF and SAND notebooks, is a Gauss-Seidel one.

This notebook builds the Gauss-Seidel cut in PROCESS's call order, shows the run order
it produces, runs it, and compares the three choices.

In [1]:
import os, sys, time
from pathlib import Path

HERE = Path.cwd()                                   # the notebook's own folder
REPO = next(p for p in (HERE, *HERE.parents) if (p / "functional_process").is_dir())
os.chdir(REPO)                                      # input files are named relative to PROCESS/
# The editable cottax checkout beside this repo, when the layout is the documented one
# (`two_opt_driver/scripts/README.md`), ahead of any installed copy.
_jaxgraph = REPO.parent.parent / "jaxgraph" / "src"
for entry in (str(REPO), str(_jaxgraph)):
    if Path(entry).is_dir() and entry not in sys.path:
        sys.path.insert(0, entry)

import jax
jax.config.update("jax_enable_x64", True)          # PROCESS is float64 throughout
import cottax
import numpy as np


print("repo  :", REPO)
print("cottax:", Path(cottax.__file__).parent)

repo  : /home/wrutten/projects/functional_PROCESS/PROCESS
cottax: /home/wrutten/projects/jaxgraph/src/cottax


## The graph

The models of the Helias stellarator input file, as the port declares them. Each node
is one model function; it reads and writes PROCESS's own variables (`.physics.rmajor`
is `data.physics.rmajor`). One disconnected node, the vacuum duct root find, is left
out. Five groups of models feed back into each other. Three models contain a solve of
their own.

In [2]:
INPUT = "tests/regression/input_files/stellarator_helias.IN.DAT"

from cottax.blocking import Blocking

from functional_process.cottax import native
from functional_process.cottax.indat import graph_for, machine_from_indat
from functional_process.cottax.mda_harness import _without_excluded
from functional_process.cottax.queries import declared
from functional_process.cottax.visualization.grouping import driver_name, problem_kind

ref = native.native_reference(INPUT)          # the file's own values -- PROCESS-free
machine_graph = graph_for(machine_from_indat(INPUT))
raw = _without_excluded(machine_graph)

print(f"{len(raw.nodes)} nodes; cyclic components of sizes {[len(c) for c in raw.cycles]}")
print("problems the models declare themselves:")
for p in declared(raw):
    print(f"   {p.spelling:55s} {problem_kind(raw[p])}")

152 nodes; cyclic components of sizes [2, 6, 2, 2, 2]
problems the models declare themselves:
   ^problem.stellarator.coils.intersect                    root-find
   ^problem.physics.profiles.ion_vol_avg_temperature       fixed-point
   ^problem.power.delta_eta_step                           fixed-point


## The recipe

`recipes.recipe("gauss_seidel")` looks at each coupled group and copies the variables
that are read before they are computed in call order. `Recipe.plan` returns the result
as a plan: the original graph plus the list of operations applied to it. The operations
are printed below, one per line, after a record per group of what was copied.

In [3]:
from functional_process.architecture_examples.notebook_tools import print_recipe
from functional_process.cottax import recipes

recipe = recipes.recipe("gauss_seidel")
plan, records = recipe.plan(raw)

for r in records:
    print(f"component of {len(r.component)} nodes: {r.n_variables} variable(s) cut over {r.n_reads} read(s)"
          + (f" -> {r.problem.spelling}" if r.problem is not None else " -> nothing to do")
          + (f"; nested inside it: {[n.spelling for n in r.nested]}" if r.nested else ""))
    for c in r.cuts:
        print(f"      cut {c.var.spelling:45s} read by {[n.spelling for n in c.readers]}")

print("\nthe plan:")
print_recipe(plan)
print("\nsame graph as recipe(raw):", plan.graph == recipe(raw))

component of 2 nodes: 0 variable(s) cut over 0 read(s) -> ^problem.physics.profiles.ion_vol_avg_temperature
component of 6 nodes: 7 variable(s) cut over 7 read(s) -> ^problem.physics.fusion_power_totals_mw.mda
      cut .physics.fusden_plasma                        read by ['.physics.fusion_totals_no_beam']
      cut .physics.fusden_plasma_alpha                  read by ['.physics.fusion_totals_no_beam']
      cut .physics.dt_power_density_plasma              read by ['.physics.fusion_power_totals_mw']
      cut .physics.dhe3_power_density                   read by ['.physics.fusion_power_totals_mw']
      cut .physics.dd_power_density                     read by ['.physics.fusion_power_totals_mw']
      cut .physics.nd_plasma_fuel_ions_vol_avg          read by ['.physics.fusion_rates']
      cut .physics.nd_plasma_ions_total_vol_avg         read by ['.physics.profiles.parameterisation.parabolic_on_axis_densities']
component of 2 nodes: 0 variable(s) cut over 0 read(s) -> nothing to do

Two groups needed copies: seven variables in the fusion-power group, one in the
first-wall group. The other three groups already contain a solve that closes their
loop, so nothing is done to them.

### Assign the drivers

`default_drivers` picks a driver by problem type: fixed-point iteration for a cut,
Newton for a root find. `assign_drivers` attaches them in the graph. The iteration is
asked to report how many sweeps it took, so the count comes out of the run like any
other value. The blocking below is the graph grouped into blocks in run order, one block
per iterated group. A schedule is that order made runnable.

In [4]:
from cottax.evaluation.schedule import Schedule

from functional_process.cottax.core.solver.drivers import PicardDriver
from functional_process.cottax.mda import assign_drivers, default_drivers

drivers = default_drivers(plan.graph)
for problem, driver in drivers.items():
    if isinstance(driver, PicardDriver):
        drivers[problem] = PicardDriver(report_steps=True)
runnable = assign_drivers(plan.graph, drivers)     # one `Assign` per problem, plus `Supply` of a start the graph computes
blocking = Blocking.scc(runnable)
schedule = Schedule(blocking)

for problem, block in zip(blocking.problems, blocking.blocks, strict=True):
    if problem is not None:
        print(f"{problem.spelling:55s} {len(block):3d} nodes  {driver_name(runnable[problem])}")
print(f"{len(blocking.blocks)} blocks, {len(schedule.steps)} schedule steps, {len(schedule.inputs)} inputs")

^problem.physics.profiles.ion_vol_avg_temperature         2 nodes  PicardDriver
^problem.physics.fusion_power_totals_mw.mda               7 nodes  PicardDriver
^problem.stellarator.coils.intersect                      2 nodes  SeededNewtonDriver
^problem.stellarator.fw_area.mda                          3 nodes  PicardDriver
^problem.power.delta_eta_step                             2 nodes  PicardDriver
143 blocks, 143 schedule steps, 310 inputs


## The process

The design structure matrix (DSM) in run order. Every model is a row. A mark below the
diagonal is a feed-forward, a mark above it a feedback. Iterated groups are boxed and
labelled with their driver. Everything else runs once, in an order derived from what
each model reads and writes. The DSM is written as an interactive page next to this notebook. In the page, hover a
cell for the variables it carries and click a box to fold it.

In [5]:
from functional_process.cottax.render_xdsm import SPELLING
from functional_process.cottax.visualization.grouping import render_grouped_dsm_html, structure_order

dsm = render_grouped_dsm_html(
    blocking, order=structure_order(blocking),
    title="stellarator_helias -- Gauss-Seidel MDA (recipe cut, call order), run order",
    file_name="dsm_mda_gauss_seidel", outdir=str(HERE), write=True, formatter=SPELLING,
)
print("written:", dsm.path)

Using adapted ragraph from debug branch
written: /home/wrutten/projects/functional_PROCESS/PROCESS/functional_process/architecture_examples/mda_gauss_seidel/dsm_mda_gauss_seidel.html


## Run it

The run starts from the state after one pass in call order, where PROCESS starts too,
and runs as one compiled program. The sweep counts come out of the result. The converged
values are compared one by one with those the hand-picked cut reaches.

In [6]:
from cottax.names import PathMap

from functional_process.cottax.sand_harness import _mda_runner, cold_state, mda_env, seed_env

env = PathMap(seed_env(ref.data, schedule, runnable, cold_state(ref.data, machine_graph)))
run = _mda_runner(schedule)

began = time.perf_counter()
out = dict(run(env))
print(f"first run {time.perf_counter() - began:.1f} s (compiles)")
began = time.perf_counter()
out = dict(run(env))
print(f"warm run  {(time.perf_counter() - began) * 1000:.1f} ms\n")

steps = {v.spelling.split("^problem", 1)[-1]: int(out[v]) for v in out if v.spelling.startswith("^driver_out.steps")}
for block, n in steps.items():
    print(f"Picard steps {n:3d}   {block}")

_, base = mda_env(ref, graph=machine_graph)         # the hand cut's fixed point
worst, compared = 0.0, 0
for var, value in out.items():
    if var.spelling.startswith("^") or var not in base:
        continue
    a, b = np.asarray(value, float), np.asarray(base[var], float)
    if a.shape != b.shape or not a.size or (np.all(a == 0) and np.all(b == 0)):
        continue
    compared += 1
    worst = max(worst, float(np.max(np.abs(a - b) / np.maximum(np.abs(b), 1e-300))))
print(f"\n{compared} variables compared with the hand cut's MDA; worst relative difference {worst:.1e}")

first run 2.1 s (compiles)
warm run  6.8 ms

Picard steps   2   .physics.profiles.ion_vol_avg_temperature
Picard steps   3   .physics.fusion_power_totals_mw.mda
Picard steps   5   .stellarator.fw_area.mda
Picard steps   2   .power.delta_eta_step



705 variables compared with the hand cut's MDA; worst relative difference 2.4e-07


## The three choices side by side

Same models, same start, same iteration; only the copied variables differ. `steps` is
the number of sweeps per group. `depth` is how many models of the group run one after
another within a sweep: one for Jacobi, the sweep length for Gauss-Seidel. So
`steps x depth` counts sequential model evaluations. `paper_tests/mda_convergence.py`
measures this for every reference input file.

In [7]:
import networkx as nx
from cottax.abstract import runnable as body_of

from functional_process.cottax.mda import cut_graph

def measure(cut_fn):
    graph = cut_fn(raw)
    drivers = default_drivers(graph)
    for problem, driver in drivers.items():
        if isinstance(driver, PicardDriver):
            drivers[problem] = PicardDriver(report_steps=True)
    runnable = assign_drivers(graph, drivers)
    blocking = Blocking.scc(runnable)
    schedule = Schedule(blocking)
    env = PathMap(seed_env(ref.data, schedule, runnable, cold_state(ref.data, machine_graph)))
    out = dict(_mda_runner(schedule)(env))
    steps = {v.spelling.split("^problem", 1)[-1]: int(out[v]) for v in out if v.spelling.startswith("^driver_out.steps")}
    depths = {}
    for sub, problem in zip(blocking.subgraphs, blocking.problems, strict=True):
        if problem is not None and len(sub.nodes) > 1:
            body = body_of(sub)
            depths[problem.spelling.split("^problem", 1)[-1]] = nx.dag_longest_path_length(body._nx_dependencies) + 1
    return {"steps": steps, "total": sum(steps.values()),
            "sequential": sum(steps.get(b, 0) * d for b, d in depths.items()), "depths": depths}

table = {"hand (mda.CUTS)": measure(cut_graph)}
for name in recipes.RECIPES:
    table[name] = measure(recipes.recipe(name))

print(f"{'cut':22s} {'blocks':>6s} {'steps':>6s} {'steps x depth':>14s}   per block")
for name, m in table.items():
    print(f"{name:22s} {len(m['steps']):6d} {m['total']:6d} {m['sequential']:14d}   {m['steps']}")

RESULT = {"steps": steps, "worst_relative_difference": worst, "compared": compared,
          "recipes": {k: {"total": v["total"], "sequential": v["sequential"]} for k, v in table.items()}}
RESULT

cut                    blocks  steps  steps x depth   per block
hand (mda.CUTS)             4     12             32   {'.physics.profiles.ion_vol_avg_temperature': 2, '.physics.proton_rate_density.cycle': 3, '.fwbs.f_ster_div_single': 5, '.power.delta_eta_step': 2}
jacobi                      4     17             17   {'.physics.profiles.ion_vol_avg_temperature': 2, '.physics.fusion_power_totals_mw.mda': 5, '.stellarator.fw_area.mda': 8, '.power.delta_eta_step': 2}
gauss_seidel                4     12             26   {'.physics.profiles.ion_vol_avg_temperature': 2, '.physics.fusion_power_totals_mw.mda': 3, '.stellarator.fw_area.mda': 5, '.power.delta_eta_step': 2}
gauss_seidel_minimal        4     11             26   {'.physics.profiles.ion_vol_avg_temperature': 2, '.physics.fusion_power_totals_mw.mda': 2, '.stellarator.fw_area.mda': 5, '.power.delta_eta_step': 2}


{'steps': {'.physics.profiles.ion_vol_avg_temperature': 2,
  '.physics.fusion_power_totals_mw.mda': 3,
  '.stellarator.fw_area.mda': 5,
  '.power.delta_eta_step': 2},
 'worst_relative_difference': 2.3689166620568197e-07,
 'compared': 705,
 'recipes': {'hand (mda.CUTS)': {'total': 12, 'sequential': 32},
  'jacobi': {'total': 17, 'sequential': 17},
  'gauss_seidel': {'total': 12, 'sequential': 26},
  'gauss_seidel_minimal': {'total': 11, 'sequential': 26}}}

Every choice reaches the same converged state. They differ in how many sweeps it takes
and how long a sweep is. The graph decides where copies are needed, the recipe decides
which variables to copy, and the table shows what that decision costs.